In [1]:
!pip install -q -U accelerate bitsandbytes peft transformers datasets evaluate rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 77.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 74.8 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 req

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    BitsAndBytesConfig
)
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

# --- 1. LOAD DATA ---
dataset = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/input/datasets/hariharannarlakanti/xl-sum-telugu/telugu_train.jsonl",
        "validation": "/kaggle/input/datasets/hariharannarlakanti/xl-sum-telugu/telugu_val.jsonl",
        "test": "/kaggle/input/datasets/hariharannarlakanti/xl-sum-telugu/telugu_test.jsonl",
    }
)

# --- 2. PREPROCESS ---
MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    inputs = tokenizer(batch["text"], max_length=384, truncation=True, padding="max_length")
    targets = tokenizer(batch["summary"], max_length=96, truncation=True, padding="max_length")
    labels = targets["input_ids"]
    labels = [[-100 if token == tokenizer.pad_token_id else token for token in label] for label in labels]
    inputs["labels"] = labels
    return inputs

tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)

# --- 3. LOAD MODEL ---
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config, 
    device_map={"": 0} 
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q", "v"], 
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, lora_config)

# --- 4. EVAL SETUP ---
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# --- 5. TRAIN ---
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-telugu-qlora",
    learning_rate=3e-4, 
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=4,   
    gradient_accumulation_steps=2,  
    num_train_epochs=3,
    predict_with_generate=True, 
    eval_strategy="epoch",          
    save_strategy="epoch",
    logging_steps=50,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,     
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,3.705992,2.138032,3.543300,0.262400,3.544200,3.555200
2,3.580695,2.144279,3.716300,0.339200,3.723900,3.718800
3,3.654338,2.140888,3.572900,0.262400,3.586200,3.583700


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentra

TrainOutput(global_step=3909, training_loss=3.6824065701496145, metrics={'train_runtime': 11801.5619, 'train_samples_per_second': 2.649, 'train_steps_per_second': 0.331, 'total_flos': 2.8241789395009536e+16, 'train_loss': 3.6824065701496145, 'epoch': 3.0})

In [7]:
# This lists everything in your main output directory
import os
print(os.listdir("./mt5-telugu-qlora"))

['checkpoint-1303', 'checkpoint-3909', 'checkpoint-2606']


In [12]:
import os
print(os.listdir("./mt5-telugu-qlora/checkpoint-3909"))

['scheduler.pt', 'tokenizer_config.json', 'README.md', 'optimizer.pt', 'training_args.bin', 'adapter_model.safetensors', 'tokenizer.json', 'trainer_state.json', 'rng_state.pth', 'adapter_config.json']


In [14]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [15]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "csebuetnlp/mT5_multilingual_XLSum"
ADAPTER_PATH = "./mt5-telugu-qlora/checkpoint-3909"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    torch_dtype="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [16]:
merged_model = model.merge_and_unload()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:683: UserWarning: Input and output embeddings are no longer tied after merging. Setting `tie_word_embeddings=False` in the model config.
  warnings.warn(


In [17]:
SAVE_PATH = "./merged_mt5_telugu"

merged_model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./merged_mt5_telugu/tokenizer_config.json',
 './merged_mt5_telugu/tokenizer.json')

In [18]:
import os
print(os.listdir(SAVE_PATH))

['model.safetensors', 'tokenizer_config.json', 'config.json', 'tokenizer.json', 'generation_config.json']


In [19]:
import shutil
shutil.make_archive("merged_mt5_telugu", "zip", SAVE_PATH)

'/kaggle/working/merged_mt5_telugu.zip'

In [20]:
from IPython.display import FileLink
FileLink("merged_mt5_telugu.zip")

/kaggle/working/merged_mt5_telugu.zip

trainable params: 0 || all params: 582,401,280 || trainable%: 0.0000
None


In [22]:
model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())

trainable params: 1,769,472 || all params: 584,170,752 || trainable%: 0.3029
None


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'csebuetnlp/mT5_multilingual_XLSum' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [26]:
import random
idx = random.randint(0, len(dataset["test"])-1)
sample = dataset["test"][idx]

article = sample["text"]
gold_summary = sample["summary"]

inputs = tokenizer(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=384
).to("cuda")

trainer.model.eval()

with torch.no_grad():
    output = trainer.model.generate(
        **inputs,
        max_new_tokens=96,
        num_beams=4,
        early_stopping=True
    )

pred_summary = tokenizer.decode(output[0], skip_special_tokens=True)

print("ARTICLE:\n", article[:1000])
print("\nGOLD SUMMARY:\n", gold_summary)
print("\nFINETUNED SUMMARY:\n", pred_summary)

[transformers] Both `max_new_tokens` (=96) and `max_length`(=84) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ARTICLE:
 ఎవరైనా టెస్టుల్లో కోవిడ్ పాజిటివ్‌గా నిర్ధరణ అయినా, లేదంటే కరోనావైరస్ సోకినవారితో సన్నిహితంగా మెలగినట్లు గుర్తించినా వారు సెల్ఫ్ ఐసోలేషన్‌లోకి వెళ్లాల్సి ఉంటుంది. అలా చేయనివారికి సెప్టెంబరు 28 నుంచి భారీ జరిమానాలు విధించనున్నారు. ఇటీవల మళ్లీ కేసులు పెరగడంతో ఈ నిర్ణయం తీసుకున్నారు. శనివారం ఒక్కరోజే ఇంగ్లండ్‌లో కొత్తగా 4,422 పాజిటివ్ కేసులు, 27 మరణాలు నమోదయ్యాయి. స్కాట్లాండ్‌లో 350, వేల్స్‌లో 212, నార్తర్న్ ఐర్లాండ్‌లో 222 నమోదయ్యాయి. ఈ జరిమానాలు వెయ్యి పౌండ్ల నుంచి 10 వేల పౌండ్ల వరకు ఉంటాయి. కరోనాతో పోరాటంలో నిబంధనలు కచ్చితంగా పాటించడమే సరైన మార్గమని ప్రధాని బోరిస్ జాన్సన్ అన్నారు. ''నిబంధనలు పాటించడం చాలా అవసరం. పాజిటివ్‌గా తేలితే నిబంధనలు కచ్చితంగా పాటించాల్సిందే. లేదంటే జరిమానాలు చెల్లించాలి'' అన్నారు. ఇప్పటివరకు కరోనావైరస్ నిబంధనలు పాటించకపోవడంతో ఇంగ్లండ్, వేల్స్‌లో 19 వేల మందికి పైగా జరిమానాలు విధించారని అటార్నీ జనరల్ చెప్పారు. అయితే, ఇందులో సగం మంది కూడా ఇంకా ఆ జరిమానాలు చెల్లించలేదు. కేసుల పెరుదుల ఇలా ఏమిటీ కొత్త రూల్స్.. * ఇంగ్లండ్‌లో నేషనల్ హెల్త్ సర్వీస్ ఎవరినైనా సెల

In [27]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to("cuda")

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=96,
        num_beams=4
    )

base_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("BASE MODEL:")
print(base_summary)

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Both `max_new_tokens` (=96) and `max_length`(=84) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL:
కరోనావైరస్ పాజిటివ్ గా నిర్ధరణ అయినా, లేదంటే సెల్ఫ్ ఐసోలేషన్ లోకి వెళ్లకుండా వారికి భారీ జరిమానాలు విధించనున్నారు.


In [28]:
model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())

trainable params: 1,769,472 || all params: 584,170,752 || trainable%: 0.3029
None


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(


In [29]:
# Load the merged model
merged_model_path = "./merged_mt5_telugu"
tokenizer = AutoTokenizer.from_pretrained(merged_model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(merged_model_path)

# Use a text from your test set
text = "మీ పరీక్షా పాఠ్యపుస్తకం" # Example text
inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs)

print("Generated Summary:", tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generated Summary: ఇవి కూడా చదవండి:
